# Flujo de trabajo manual en ML

A manera de ejemplo, se va a mostrar como se implementaría un flujo de trabajo manual para entrenar un modelo de ML, que incluiría los siguientes pasos:
- Partición de los datos en subconjuntos de entrenamiento y prueba.
- Procesamiento por separado de las características cualitativas y cuantitativas en los conjuntos de entrenamiento y prueba.
- Concatenación de los procesos para formar las matrices de características.
- Entrenamiento del modelo, y evaluación del mismo.
- Uso del modelo entrenado para hacer predicciones.

## Ejemplo de uso

Se va a entrenar un modelo con el dataset **adult.data**, que tiene como objetivo predecir la variable **income**.

Sa va a hacer el siguiente preprocesamiento a las variables del dataset:

* **age** y **hours-per-week** se van a estandarizar con `StandardScaler`.
* **fnlwgt** se va a transformar con `PowerTransformer`.
* **education** se va a codificar con `OrdinalEncoder`.
* **marital-status**, **occupation** y **sex** se codificarán mediante `OneHotEncoder`.
* Las otras variables se van a descartar del modelo.

Empezamos cargando el dataset, y por facilidad, eliminando los datos nulos y duplicados.

In [1]:
import pandas as pd

df = pd.read_csv(
    "http://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    header =None,
    names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
             'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
             'hours-per-week', 'native-country', 'income'],
    na_values= [' ?']
    )

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Paso 1: Partición de los datos

A continuación, descartamos las variables que no vamos a usar, y partimos los datos en subconjuntos de entrenamiento y prueba.

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(
    ['income', 'workclass', 'education-num', 'relationship', 'capital-gain',
     'capital-loss', 'native-country', 'race'],
    axis=1
    )

y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.8)

print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (24111, 7)
Tamaño del conjunto de prueba es: (6028, 7)


In [6]:
X_train

,age,fnlwgt,education,marital-status,occupation,sex,hours-per-week
26312,28,293398,HS-grad,Separated,Sales,Female,40
23791,34,293900,11th,Married-spouse-absent,Craft-repair,Male,55
14775,59,233312,Some-college,Married-civ-spouse,Craft-repair,Male,40
14866,59,159724,Masters,Married-civ-spouse,Sales,Male,55
25476,51,96190,Some-college,Married-civ-spouse,Adm-clerical,Female,40
...,...,...,...,...,...,...,...
17289,27,120155,HS-grad,Married-civ-spouse,Adm-clerical,Male,39
5192,33,192644,HS-grad,Separated,Handlers-cleaners,Male,35
12172,30,189759,Bachelors,Never-married,Transport-moving,Male,40
235,42,303044,HS-grad,Married-civ-spouse,Farming-fishing,Male,40


### Paso 2: Procesar por separado cada característica de los conjuntos de entrenamiento y prueba:

Empezamos separando las características por tipo de procesamiento a aplicar:

In [10]:
X_train_age_hours = X_train[['age', 'hours-per-week']]
X_test_age_hours = X_test[['age', 'hours-per-week']]

X_train_fnlwgt = X_train[['fnlwgt']]
X_test_fnlwgt = X_test[['fnlwgt']]

X_train_education = X_train[['education']]
X_test_education = X_test[['education']]

X_train_marital_occupation_sex = X_train[['marital-status', 'occupation', 'sex']]
X_test_marital_occupation_sex = X_test[['marital-status', 'occupation', 'sex']]

Ahora, creamos los transformadores que se van a usar:

In [12]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PowerTransformer

ss = StandardScaler() # Para preprocesar age y hours-per-week
pt = PowerTransformer() # Para preprocesar fnlwgt
orden = [' Preschool', ' 1st-4th', ' 5th-6th', ' 7th-8th', ' 9th', ' 10th', ' 11th', ' 12th',
         ' HS-grad',' Some-college', ' Prof-school', ' Assoc-acdm', ' Assoc-voc', ' Bachelors',
         ' Masters', ' Doctorate']
ore = OrdinalEncoder(categories=[orden], dtype='int') # Para preprocesar education
ohe = OneHotEncoder(sparse_output=False, drop='if_binary') # marital-status, occupation y sex

Luego, entrenamos los transformadores con los datos de entrenamiento:

In [13]:
ss.fit(X_train_age_hours)
pt.fit(X_train_fnlwgt)
ore.fit(X_train_education)
ohe.fit(X_train_marital_occupation_sex)

OneHotEncoder(drop='if_binary', sparse_output=False)

Luego, transformamos los datos de entrenamiento y prueba por separado:

In [14]:
X_train_age_hours = ss.transform(X_train_age_hours)
X_test_age_hours = ss.transform(X_test_age_hours)

X_train_fnlwgt = pt.transform(X_train_fnlwgt)
X_test_fnlwgt = pt.transform(X_test_fnlwgt)

X_train_education = ore.transform(X_train_education)
X_test_education = ore.transform(X_test_education)

X_train_marital_occupation_sex = ohe.transform(X_train_marital_occupation_sex)
X_test_marital_occupation_sex = ohe.transform(X_test_marital_occupation_sex)

### Paso 3: Concatenación de las matrices de características procesadas:

In [17]:
import numpy as np

X_train = np.concatenate(
    [X_train_age_hours, X_train_fnlwgt, X_train_education, X_train_marital_occupation_sex],
    axis=1
    )

X_test = np.concatenate(
    [X_test_age_hours, X_test_fnlwgt, X_test_education, X_test_marital_occupation_sex],
    axis=1
    )

print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (24111, 26)
Tamaño del conjunto de prueba es: (6028, 26)


### Paso 4: Entrenamiento del modelo

Vamos a entrenar un modelo lineal de regresión logística:

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=10000)

Y lo evaluamos:

In [20]:
print(f'Exactitud del modelo en el conjunto de entrenamiento: {model.score(X_train, y_train)}')
print(f'Exactitud del modelo en el conjunto de prueba: {model.score(X_test, y_test)}')

Exactitud del modelo en el conjunto de entrenamiento: 0.826427771556551
Exactitud del modelo en el conjunto de prueba: 0.8173523556735236


### Paso 5: Usar el modelo para hacer predicciones:

Supongamos que queremos hacer una predicción con los siguientes datos:
- age: 50
- hours-per-week: 45
- fnlwgt: 100000
- education: Masters
- marital-status: Married-civ-spouse
- occupation: Sales
- sex: Male

Primero, deberíamos crear un dataframe con estos datos, luego procesarlo, y finalmente hacer la predicción:

In [21]:
data_new = [50, 45, 100000, ' Masters', ' Married-civ-spouse', ' Sales', ' Male']
data_new = pd.DataFrame(data_new).T
data_new.columns = ['age', 'hours-per-week', 'fnlwgt', 'education', 'marital-status', 'occupation', 'sex']
data_new

,age,hours-per-week,fnlwgt,education,marital-status,occupation,sex
0,50,45,100000,Masters,Married-civ-spouse,Sales,Male


In [23]:
X_new_age_hours = ss.transform(data_new[['age', 'hours-per-week']])
X_new_fnlwgt = pt.transform(data_new[['fnlwgt']])
X_new_education = ore.transform(data_new[['education']])
X_new_marital_occupation_sex = ohe.transform(data_new[['marital-status', 'occupation', 'sex']])
X_new = np.concatenate(
    [X_new_age_hours, X_new_fnlwgt, X_new_education, X_new_marital_occupation_sex],
    axis=1
    )
X_new

array([[ 0.8882938 ,  0.33833344, -0.85301181, 14.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ]])

In [24]:
y_new = model.predict(X_new)
y_new

array([' >50K'], dtype=object)